# KI-4: Filter Experiment (Channel Quality Impact)

## 실험 개요

**연구 질문:** 채널 품질(토큰 제한 + 손상률)이 전문가 에이전트의 응답 품질에 미치는 영향은? 전문성과 채널 품질을 함께 고려한 에이전트 선택이 유효한가?

### 에이전트 구성
- **4 Specialist Agents** (GPT-4o-mini): Medical, Legal, Finance, Tech 각 분야 전문가
- **Grader** (GPT-4o): 0-10점 척도로 응답을 채점
- **Receiving Agent: 없음** - 손상된 텍스트가 직접 채점됨 (AI 복원 단계 없음)

### 채널 설정 (Harsh)
| Channel | max_tokens (API 강제) | Corruption Rate |
|---------|----------------------|----------------|
| Good | 500 | 0% |
| Medium | 150 | 20% |
| Bad | 60 | 50% |

### 실험 조건 (5가지)
| 조건 | 에이전트 선택 기준 |
|------|------------------|
| **Fixed Order** | 항상 Medical -> Legal 순서로 첫 2명 선택 |
| **Expertise Only** | 전문성 점수 상위 2명 |
| **Channel Only** | 채널 품질 상위 2명 |
| **Joint** | 전문성 x 채널 품질 상위 2명 |
| **Joint + Early Stop** | Joint과 동일하나, 1등이 primary expert + Good 채널이면 1명만 사용 |

### 문제 설계
- 15문제: 의료(4), 법률(4), 재무(4), 기술(3)
- 각 문제에 primary/secondary domain 지정
- 채널 할당: seed=42 기반 결정적 랜덤

### 핵심: 채널 손상 메커니즘
- `max_tokens`로 API 레벨에서 강제 절단
- corruption: 각 단어를 corruption_rate 확률로 `[???]`로 치환
- 손상된 텍스트가 복원 없이 직접 채점됨

### 기대 결과
- Joint 조건이 Fixed/Expertise/Channel 단독보다 우수
- Early Stop은 Joint과 유사하면서 API 비용 절감

In [ ]:
# ============================================================
# Setup: API Key and Dependencies
# ============================================================
import os
import json
import time
import re
import random
import math
from datetime import datetime

import requests
import pandas as pd
import numpy as np

# Load API key from environment variable or prompt for input
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    OPENAI_API_KEY = input("OpenAI API Key를 입력하세요: ").strip()

assert OPENAI_API_KEY, "API Key가 설정되지 않았습니다."

API_URL = "https://api.openai.com/v1/chat/completions"
SPECIALIST_MODEL = "gpt-4o-mini"
GRADER_MODEL = "gpt-4o"

print(f"Specialist Model: {SPECIALIST_MODEL}")
print(f"Grader Model: {GRADER_MODEL}")
print("API Key: OK (loaded)")

In [ ]:
# ============================================================
# Common Functions: API Call, Corruption, Scoring
# ============================================================

# Channel configurations (harsh settings)
CHANNELS = {
    "Good":   {"max_tokens": 500, "corruption_rate": 0.00},
    "Medium": {"max_tokens": 150, "corruption_rate": 0.20},
    "Bad":    {"max_tokens": 60,  "corruption_rate": 0.50},
}

DOMAINS = ["Medical", "Legal", "Finance", "Tech"]

SPECIALIST_PROMPTS = {
    "Medical": "You are a medical specialist. Answer the following medical question accurately and concisely. Provide specific treatments, dosages, and protocols where relevant.",
    "Legal": "You are a legal specialist. Answer the following legal question accurately and concisely. Cite specific legal principles and elements.",
    "Finance": "You are a finance specialist. Answer the following finance question accurately and concisely. Show calculations where needed.",
    "Tech": "You are a technology specialist. Answer the following computer science/technology question accurately and concisely. Be specific about algorithms and mechanisms.",
}


def call_openai(model: str, system_prompt: str, user_prompt: str,
                max_tokens: int = 1000, temperature: float = 0) -> str:
    """Call OpenAI API with explicit max_tokens control."""
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {OPENAI_API_KEY}",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    resp = requests.post(API_URL, headers=headers, json=payload, timeout=120)
    resp.raise_for_status()
    data = resp.json()
    return data["choices"][0]["message"]["content"]


def corrupt_text(text: str, rate: float) -> str:
    """Replace each word with [???] at the given corruption rate."""
    if rate == 0:
        return text
    words = text.split()
    corrupted = ["[???]" if random.random() < rate else w for w in words]
    return " ".join(corrupted)


def get_expertise(specialist_domain: str, q_primary: str, q_secondary: str) -> float:
    """Score specialist expertise for a given question."""
    if specialist_domain == q_primary:
        return 1.0
    if specialist_domain == q_secondary:
        return 0.5
    return 0.2


def get_channel_score(channel_name: str) -> float:
    """Numeric score for channel quality."""
    return {"Good": 1.0, "Medium": 0.5, "Bad": 0.1}[channel_name]


def grade_response(question: str, reference: str, corrupted_text: str) -> dict:
    """Use GPT-4o to grade the (possibly corrupted) response on 0-10 scale."""
    prompt = f"""You are a strict academic grader. Score the following answer on a scale of 0-10.

QUESTION: {question}
REFERENCE ANSWER: {reference}

RECEIVED ANSWER (this is exactly what the system received -- may be corrupted or truncated):
{corrupted_text}

Scoring criteria:
- 0-2: Answer is mostly corrupted, unreadable, or completely wrong
- 3-4: Some relevant content but major gaps or corruption makes it largely unusable
- 5-6: Partially correct, some key points present but significant issues
- 7-8: Mostly correct with minor gaps or issues
- 9-10: Comprehensive, accurate, clearly communicated

Respond with ONLY a JSON object: {{"score": <number>, "reason": "<brief explanation>"}}"""

    resp_text = call_openai(GRADER_MODEL, "You are a strict grader. Output only valid JSON.",
                            prompt, max_tokens=200, temperature=0)
    try:
        parsed = json.loads(resp_text)
        return parsed
    except json.JSONDecodeError:
        match = re.search(r"(\d+\.?\d*)", resp_text)
        return {"score": float(match.group(1)) if match else 0, "reason": resp_text}


print("Common functions loaded.")
print(f"Channels: {CHANNELS}")

## 문제 목록 + 채널 할당

15개 문제는 4개 도메인에 걸쳐 있으며, 각 문제에 primary/secondary 도메인이 지정됩니다.
채널 할당은 seed=42 기반 결정적 PRNG로 고정됩니다.

In [ ]:
# ============================================================
# Question Definitions + Deterministic Channel Assignment
# ============================================================

QUESTIONS = [
    {"id": 1,  "q": "What is the first-line treatment for hypertension in a patient with diabetes?",
     "ref": "ACE inhibitor or ARB", "primary": "Medical", "secondary": "Finance"},
    {"id": 2,  "q": "What is the differential diagnosis for acute right lower quadrant abdominal pain?",
     "ref": "Appendicitis, ovarian torsion, ectopic pregnancy, Crohn's", "primary": "Medical", "secondary": "Legal"},
    {"id": 3,  "q": "Describe the management protocol for acute ST-elevation myocardial infarction.",
     "ref": "PCI within 90 minutes, dual antiplatelet therapy, heparin", "primary": "Medical", "secondary": "Tech"},
    {"id": 4,  "q": "What is the drug of choice for status epilepticus and subsequent management?",
     "ref": "IV lorazepam first-line, then IV phenytoin/fosphenytoin", "primary": "Medical", "secondary": "Legal"},
    {"id": 5,  "q": "What are the four elements required to prove negligence in tort law?",
     "ref": "Duty, breach, causation, damages", "primary": "Legal", "secondary": "Medical"},
    {"id": 6,  "q": "Explain the difference between copyright and trademark protection.",
     "ref": "Copyright protects creative expression; trademark protects brand identity/source identifiers", "primary": "Legal", "secondary": "Tech"},
    {"id": 7,  "q": "What makes a contract void versus voidable?",
     "ref": "Void: illegal purpose or lack of capacity from inception; Voidable: misrepresentation, duress, undue influence", "primary": "Legal", "secondary": "Finance"},
    {"id": 8,  "q": "What remedies are available to an employee upon wrongful termination?",
     "ref": "Back pay, reinstatement, compensatory and punitive damages", "primary": "Legal", "secondary": "Finance"},
    {"id": 9,  "q": "Calculate the NPV of a $100,000 investment that returns $30,000 per year for 5 years at a 10% discount rate.",
     "ref": "$13,724 (NPV = -100000 + 30000*PV annuity factor)", "primary": "Finance", "secondary": "Tech"},
    {"id": 10, "q": "Explain the difference between operating leverage and financial leverage.",
     "ref": "Operating leverage: proportion of fixed operating costs; Financial leverage: use of debt to amplify returns", "primary": "Finance", "secondary": "Legal"},
    {"id": 11, "q": "Calculate WACC given: 60% equity at 12% cost, 40% debt at 6% cost, corporate tax rate 25%.",
     "ref": "WACC = 0.6*12% + 0.4*6%*(1-0.25) = 7.2% + 1.8% = 9.0%", "primary": "Finance", "secondary": "Tech"},
    {"id": 12, "q": "Perform break-even analysis: fixed costs $500,000, variable cost $30/unit, selling price $80/unit.",
     "ref": "Break-even = 500000 / (80-30) = 10,000 units", "primary": "Finance", "secondary": "Medical"},
    {"id": 13, "q": "What is the time complexity of binary search and why?",
     "ref": "O(log n) because each comparison halves the search space", "primary": "Tech", "secondary": "Finance"},
    {"id": 14, "q": "How does TCP ensure reliable data delivery over an unreliable network?",
     "ref": "Sequence numbers, acknowledgments, retransmission, flow control, congestion control", "primary": "Tech", "secondary": "Legal"},
    {"id": 15, "q": "Explain the CAP theorem with practical examples of each tradeoff.",
     "ref": "Consistency, Availability, Partition tolerance -- can only guarantee 2 of 3. CP: MongoDB; AP: Cassandra; CA: traditional RDBMS (no partition)", "primary": "Tech", "secondary": "Medical"},
]

# Deterministic PRNG (matches the JS seededRandom(42) from the original script)
def seeded_random(seed):
    s = seed
    def next_val():
        nonlocal s
        s = (s * 1664525 + 1013904223) & 0xFFFFFFFF
        return (s >> 0) / 0xFFFFFFFF
    return next_val

rng = seeded_random(42)
CHANNEL_NAMES = ["Good", "Medium", "Bad"]

# Assign channels: channel_map[question_id][domain] = channel_name
channel_map = {}
for q in QUESTIONS:
    channel_map[q["id"]] = {}
    for d in DOMAINS:
        idx = int(rng() * 3)
        channel_map[q["id"]][d] = CHANNEL_NAMES[idx]

# Display channel assignments
print("Channel Assignments (seed=42):")
ch_rows = []
for q in QUESTIONS:
    cm = channel_map[q["id"]]
    ch_rows.append({"Q#": q["id"], "Primary": q["primary"],
                    "Medical": cm["Medical"], "Legal": cm["Legal"],
                    "Finance": cm["Finance"], "Tech": cm["Tech"]})
df_channels = pd.DataFrame(ch_rows)
display(df_channels)

In [ ]:
# ============================================================
# Condition Definitions: 5 specialist selection strategies
# ============================================================

def select_fixed_order(question, q_channels):
    """Always pick Medical and Legal (first 2 in fixed order)."""
    return ["Medical", "Legal"]


def select_expertise_only(question, q_channels):
    """Pick top 2 by expertise score."""
    scored = [(d, get_expertise(d, question["primary"], question["secondary"])) for d in DOMAINS]
    scored.sort(key=lambda x: -x[1])
    return [s[0] for s in scored[:2]]


def select_channel_only(question, q_channels):
    """Pick top 2 by channel quality."""
    scored = [(d, get_channel_score(q_channels[d])) for d in DOMAINS]
    scored.sort(key=lambda x: -x[1])
    return [s[0] for s in scored[:2]]


def select_joint(question, q_channels):
    """Pick top 2 by expertise * channel score."""
    scored = [(d, get_expertise(d, question["primary"], question["secondary"]) * get_channel_score(q_channels[d]))
              for d in DOMAINS]
    scored.sort(key=lambda x: -x[1])
    return [s[0] for s in scored[:2]]


def select_joint_early_stop(question, q_channels):
    """Joint selection, but only 1 agent if top pick is primary expert + Good channel."""
    scored = []
    for d in DOMAINS:
        exp = get_expertise(d, question["primary"], question["secondary"])
        ch = q_channels[d]
        scored.append((d, exp, ch, exp * get_channel_score(ch)))
    scored.sort(key=lambda x: -x[3])
    first = scored[0]
    if first[1] == 1.0 and first[2] == "Good":  # primary expert + Good channel
        return [first[0]]  # early stop: 1 agent only
    return [s[0] for s in scored[:2]]


CONDITIONS = [
    {"name": "Fixed Order",       "select": select_fixed_order},
    {"name": "Expertise Only",    "select": select_expertise_only},
    {"name": "Channel Only",      "select": select_channel_only},
    {"name": "Joint",             "select": select_joint},
    {"name": "Joint + Early Stop", "select": select_joint_early_stop},
]

print(f"Defined {len(CONDITIONS)} conditions:")
for c in CONDITIONS:
    print(f"  - {c['name']}")

In [ ]:
# ============================================================
# Trial Runner: Consult specialists -> corrupt -> grade
# ============================================================

def consult_specialist(domain: str, question: str, channel_name: str) -> dict:
    """Call a specialist through the specified channel (token limit + corruption)."""
    ch = CHANNELS[channel_name]
    raw = call_openai(SPECIALIST_MODEL, SPECIALIST_PROMPTS[domain],
                      question, max_tokens=ch["max_tokens"], temperature=0)
    corrupted = corrupt_text(raw, ch["corruption_rate"])
    return {"domain": domain, "channel": channel_name, "raw": raw, "corrupted": corrupted}


def run_question(condition, question):
    """Run a single question under one condition."""
    q_channels = channel_map[question["id"]]
    specialists = condition["select"](question, q_channels)

    # Consult each specialist through their channel
    consultations = []
    for domain in specialists:
        result = consult_specialist(domain, question["q"], q_channels[domain])
        consultations.append(result)

    # Concatenate corrupted responses (no recovery agent!)
    combined_text = "\n\n".join(
        f"[Agent {i+1} - {c['domain']}]: {c['corrupted']}"
        for i, c in enumerate(consultations)
    )

    # Grade the corrupted combined text directly
    grade = grade_response(question["q"], question["ref"], combined_text)

    return {
        "question_id": question["id"],
        "question": question["q"],
        "reference": question["ref"],
        "primary": question["primary"],
        "specialists": [
            {"domain": d, "channel": q_channels[d],
             "expertise": get_expertise(d, question["primary"], question["secondary"])}
            for d in specialists
        ],
        "score": grade["score"],
        "reason": grade.get("reason", ""),
        "num_agents": len(specialists),
    }


print("Trial runner loaded.")

## Condition 1: Fixed Order

항상 Medical -> Legal 순서로 첫 2명의 전문가를 선택합니다.
질문의 도메인이나 채널 품질을 전혀 고려하지 않는 기준선(baseline)입니다.

In [ ]:
# ============================================================
# Condition 1: Fixed Order
# ============================================================
print("Running Condition 1: Fixed Order...")
results_fixed = []
for q in QUESTIONS:
    print(f"  Q{q['id']}: {q['primary']}")
    result = run_question(CONDITIONS[0], q)
    results_fixed.append(result)
    print(f"    -> score={result['score']}, agents={[s['domain'] for s in result['specialists']]}")
    time.sleep(0.5)

avg = np.mean([r["score"] for r in results_fixed])
print(f"\nFixed Order: Avg Score = {avg:.2f}")

## Condition 2: Expertise Only

전문성 점수가 가장 높은 2명의 전문가를 선택합니다.
채널 품질은 무시합니다.

In [ ]:
# ============================================================
# Condition 2: Expertise Only
# ============================================================
print("Running Condition 2: Expertise Only...")
results_expertise = []
for q in QUESTIONS:
    print(f"  Q{q['id']}: {q['primary']}")
    result = run_question(CONDITIONS[1], q)
    results_expertise.append(result)
    print(f"    -> score={result['score']}, agents={[s['domain'] for s in result['specialists']]}")
    time.sleep(0.5)

avg = np.mean([r["score"] for r in results_expertise])
print(f"\nExpertise Only: Avg Score = {avg:.2f}")

## Condition 3: Channel Only

채널 품질이 가장 좋은 2명의 전문가를 선택합니다.
전문성은 무시합니다.

In [ ]:
# ============================================================
# Condition 3: Channel Only
# ============================================================
print("Running Condition 3: Channel Only...")
results_channel = []
for q in QUESTIONS:
    print(f"  Q{q['id']}: {q['primary']}")
    result = run_question(CONDITIONS[2], q)
    results_channel.append(result)
    print(f"    -> score={result['score']}, agents={[s['domain'] for s in result['specialists']]}")
    time.sleep(0.5)

avg = np.mean([r["score"] for r in results_channel])
print(f"\nChannel Only: Avg Score = {avg:.2f}")

## Condition 4: Joint

전문성 x 채널 품질의 결합 점수로 상위 2명을 선택합니다.
두 요소를 모두 고려하여 최적의 전문가를 선택하는 전략입니다.

In [ ]:
# ============================================================
# Condition 4: Joint (Expertise * Channel)
# ============================================================
print("Running Condition 4: Joint...")
results_joint = []
for q in QUESTIONS:
    print(f"  Q{q['id']}: {q['primary']}")
    result = run_question(CONDITIONS[3], q)
    results_joint.append(result)
    print(f"    -> score={result['score']}, agents={[s['domain'] for s in result['specialists']]}")
    time.sleep(0.5)

avg = np.mean([r["score"] for r in results_joint])
print(f"\nJoint: Avg Score = {avg:.2f}")

## Condition 5: Joint + Early Stop

Joint과 동일한 선택 기준이지만, 1등 전문가가 primary expert이면서 Good 채널이면
1명만 사용합니다 (early stopping). API 비용을 절감하면서 품질을 유지하는 전략입니다.

In [ ]:
# ============================================================
# Condition 5: Joint + Early Stop
# ============================================================
print("Running Condition 5: Joint + Early Stop...")
results_early = []
for q in QUESTIONS:
    print(f"  Q{q['id']}: {q['primary']}")
    result = run_question(CONDITIONS[4], q)
    results_early.append(result)
    agents_str = [s['domain'] for s in result['specialists']]
    es_tag = " [EARLY STOP]" if result["num_agents"] == 1 else ""
    print(f"    -> score={result['score']}, agents={agents_str}{es_tag}")
    time.sleep(0.5)

avg = np.mean([r["score"] for r in results_early])
print(f"\nJoint + Early Stop: Avg Score = {avg:.2f}")

## 결과 요약

In [ ]:
# ============================================================
# Results Summary: Table + Analysis + JSON Export
# ============================================================

all_condition_results = {
    "Fixed Order": results_fixed,
    "Expertise Only": results_expertise,
    "Channel Only": results_channel,
    "Joint": results_joint,
    "Joint + Early Stop": results_early,
}

# Summary table
summary_rows = []
for name, results in all_condition_results.items():
    scores = [r["score"] for r in results]
    sorted_scores = sorted(scores)
    n = len(sorted_scores)
    median = (sorted_scores[n//2 - 1] + sorted_scores[n//2]) / 2 if n % 2 == 0 else sorted_scores[n//2]
    avg_agents = np.mean([r["num_agents"] for r in results])
    summary_rows.append({
        "Condition": name,
        "Avg Score": round(np.mean(scores), 2),
        "Median": round(median, 2),
        "Min": round(min(scores), 2),
        "Max": round(max(scores), 2),
        "StdDev": round(np.std(scores), 2),
        "Avg Agents": round(avg_agents, 2),
    })

df_summary = pd.DataFrame(summary_rows)
print("=" * 80)
print("                    KI-4 RESULTS SUMMARY")
print("=" * 80)
display(df_summary)

# Per-question breakdown
print("\n--- Per-Question Scores ---")
per_q_rows = []
for q in QUESTIONS:
    row = {"Q#": q["id"], "Domain": q["primary"]}
    for name, results in all_condition_results.items():
        r = [x for x in results if x["question_id"] == q["id"]][0]
        row[name] = r["score"]
    per_q_rows.append(row)

df_per_q = pd.DataFrame(per_q_rows)
display(df_per_q)

# Key analysis
print("\n--- Key Analysis ---")
avgs = {row["Condition"]: row["Avg Score"] for row in summary_rows}
print(f"  Joint vs Fixed Order:    {avgs['Joint'] - avgs['Fixed Order']:+.2f}")
print(f"  Joint vs Expertise Only: {avgs['Joint'] - avgs['Expertise Only']:+.2f}")
print(f"  Joint vs Channel Only:   {avgs['Joint'] - avgs['Channel Only']:+.2f}")
print(f"  Early Stop vs Joint:     {avgs['Joint + Early Stop'] - avgs['Joint']:+.2f}")
print(f"  Score range:             {min(avgs.values()):.2f} - {max(avgs.values()):.2f}")

# Save results to JSON
output = {
    "experiment": "KI-4 Filter: No Recovery, Harsh Channels",
    "timestamp": datetime.now().isoformat(),
    "design": {
        "channels": CHANNELS,
        "max_consultations": 2,
        "no_recovery_agent": True,
        "grader_model": GRADER_MODEL,
        "specialist_model": SPECIALIST_MODEL,
    },
    "channel_assignments": {str(k): v for k, v in channel_map.items()},
    "summary": summary_rows,
    "per_question": per_q_rows,
}

out_path = "ki4_filter_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"\nResults saved to {out_path}")